In [ ]:
import pickle
import re
import string

from google.colab import drive, files
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from wordcloud import WordCloud
import xgboost as xgb

In [ ]:
nltk.download('punkt')  # For word_tokenize
nltk.download('stopwords')  # For stopwords
nltk.download('wordnet')

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

def find_file(name, path):
    for root, dirs, files in os.walk(path):
        if name in files:
            return os.path.join(root, name)
    return None

real_path = find_file('Reviews.csv', '/content/drive/MyDrive')

if real_path:
    print(f" {real_path}")
else:
    print("not found")
df = pd.read_csv(real_path)

df.head()

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

In [ ]:

ax = df['Score'].value_counts().sort_index() \
    .plot(kind='bar',
          title='Count of Reviews by Stars',
          figsize=(10, 5))

ax.set_xlabel('Review Stars')
plt.show()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df_copy = df.copy()

df = df[['Text', 'Summary', 'Score']]
df = df.drop_duplicates(subset=['Text'], keep='first')
df.head()

In [ ]:
df.duplicated().sum()

In [ ]:
def cleaning(text):



    text = re.sub(r'https?:\/\/\S+', ' ', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'@[A-Za-z0-9_]+', ' ', text)
    text = re.sub(r'#[A-Za-z0-9_]+', ' ', text)
    text=re.sub(r'<.*?>',' ',text)
    text=text.lower()
    text = text.split()
    text = ' '.join(text)
    return text



In [ ]:
df['cleaning']=df['Text'].apply(cleaning)

In [ ]:
import string
punc=string.punctuation
def remove_punc(text):
    return text.translate(str.maketrans('','',punc))


In [ ]:
df['Remove_punc']=df['cleaning'].apply(remove_punc)

In [ ]:
df.head()

In [ ]:
nltk.download('punkt_tab')

In [ ]:
### word tokenization
from nltk.tokenize import word_tokenize
df['tokenized']=df['cleaning'].apply(word_tokenize)

In [ ]:
df.head()

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

important_words = {
    'not', 'no', 'nor', 'never',
    'very', 'too', 'so',
    'but', 'however'
}

custom_stopwords = stop_words - important_words

def remove_stopWords(words):
    return [word for word in words if word.lower() not in custom_stopwords]

In [ ]:
df['Nostopword_text']=df['tokenized'].apply(remove_stopWords)

In [ ]:
df.head()

In [ ]:
from nltk.stem import WordNetLemmatizer
lemmatizer=WordNetLemmatizer()
def lemmatizer_word(text):
 lemmas=[lemmatizer.lemmatize(word,pos='v') for word in text  ]
 return lemmas
df['lemmatizer_text']=df['Nostopword_text'].apply(lemmatizer_word)


In [ ]:
df.head()

In [ ]:
df['clean_text'] = df['lemmatizer_text'].apply(lambda tokens: " ".join(tokens))


In [ ]:
for text in df['Text'][3000:3005]:
    print(text)
    print('-' * 40)

In [ ]:
for text in df['clean_text'][3000:3005]:
    print(text)
    print('-' * 40)

In [ ]:
text = " ".join(df['clean_text'])

In [ ]:
text

In [ ]:
# wordcloud = WordCloud(width=800, height=400).generate(text)

# plt.imshow(wordcloud)
# plt.axis('off')
# plt.title("Overall Word Cloud")
# plt.show()

In [ ]:
# df.drop(['cleaning','Remove_punc','tokenized','Nostopword_text','lemmatizer_text'],axis=1,inplace=True)

In [ ]:
df.head()

In [ ]:
import pandas as pd

required_columns = ['Time', 'Score', 'Text', 'HelpfulnessNumerator', 'HelpfulnessDenominator']
if all(col in df.columns for col in required_columns):

    df['Time'] = pd.to_datetime(df['Time'], unit='s')
    df['ReviewYear'] = df['Time'].dt.year
    df['ReviewMonth'] = df['Time'].dt.month
    df['DayOfWeek'] = df['Time'].dt.dayofweek
else:
    print("Missing columns! Please check: ", df.columns)

In [ ]:
import string

df['text_length'] = df['Text'].str.len()
df['word_count'] = df['Text'].apply(lambda x: len(str(x).split()))
df['avg_word_length'] = df['text_length'] / (df['word_count'] + 1)
df['avg_sentence_length'] = df['text_length'] / df['Text'].apply(lambda x: len(str(x).split('.')))

In [ ]:
df['punct_count'] = df['Text'].apply(lambda x: len([c for c in str(x) if c in string.punctuation]))
df['punct_percent'] = df['punct_count'] / (df['text_length'] + 1)
df['caps_ratio'] = df['Text'].apply(lambda x: sum(1 for c in str(x) if c.isupper()) / (len(str(x)) + 1))

In [ ]:
df['exclamation_count'] = df['Text'].str.count('!')
df['question_count'] = df['Text'].str.count('\?')

In [ ]:
# from textblob import TextBlob

# sentiments = [TextBlob(str(text)).sentiment for text in df['Text']]

# df['polarity'] = [s.polarity for s in sentiments]
# df['subjectivity'] = [s.subjectivity for s in sentiments]

In [ ]:
df['lexical_richness'] = df['Text'].apply(lambda x: len(set(str(x).split())) / len(str(x).split()) if len(str(x).split()) > 0 else 0)
df['summary_length'] = df['Summary'].str.len()
df['summary_ratio'] = df['summary_length'] / (df['text_length'] + 1)

In [ ]:
cols_to_show = ['text_length', 'avg_sentence_length', 'caps_ratio', 'lexical_richness']
print(df[cols_to_show].head())

In [ ]:
# from gensim.models.fasttext import load_facebook_model
# import numpy as np
# import pandas as pd

# model = load_facebook_model('cc.en.300.bin')

# def get_fasttext_features(text):
#     words = str(text).lower().split()
#     vectors = [model.wv[word] for word in words if word in model.wv]
#     if not vectors:
#         return np.zeros(model.vector_size)
#     return np.mean(vectors, axis=0)

# fasttext_vectors = df['Text'].apply(get_fasttext_features)
# ft_matrix = np.vstack(fasttext_vectors.values)
# ft_df = pd.DataFrame(ft_matrix, index=df.index).add_prefix('ft_')

# df_final = pd.concat([
#     df[['lexical_richness', 'summary_ratio', 'polarity']],
#     ft_df
# ], axis=1)

In [ ]:
df.sample()

In [ ]:
import numpy as np

def map_scores_to_classes(score):
    if score in [1, 2]:
        return 0  # Negative
    elif score == 3:
        return 1  # Neutral
    elif score in [4, 5]:
        return 2  # Positive
    return -1

df['Mapped_Score'] = df['Score'].apply(map_scores_to_classes)


In [ ]:
extra_cols = ['lexical_richness', 'summary_ratio', 'text_length', 'caps_ratio']

X_text = df['clean_text'].apply(lambda x: str(x).split())
X_extra = df[extra_cols]
y = df['Mapped_Score']

print("Original Score Value Counts:")
print(df['Score'].value_counts().sort_index())
print("\nMapped Class Value Counts:")
print(y.value_counts().sort_index())

X_text_train, X_text_test, X_extra_train, X_extra_test, y_train, y_test = train_test_split(
    X_text, X_extra, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=10000)

X_ft_train = tfidf_vectorizer.fit_transform(X_text_train.apply(lambda x: ' '.join(x)))
X_ft_test = tfidf_vectorizer.transform(X_text_test.apply(lambda x: ' '.join(x)))

In [ ]:
from scipy.sparse import hstack

In [ ]:
scaler = StandardScaler()

X_extra_train_scaled = scaler.fit_transform(X_extra_train)
X_extra_test_scaled  = scaler.transform(X_extra_test)

X_train_final = hstack([X_ft_train, X_extra_train_scaled])
X_test_final  = hstack([X_ft_test, X_extra_test_scaled])

lr_model = LogisticRegression(
    max_iter=2000,
    multi_class='multinomial',
    solver='lbfgs',
    class_weight='balanced'
)

print("Training Logistic Regression with balanced weights...")
lr_model.fit(X_train_final, y_train)

y_pred = lr_model.predict(X_test_final)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
with open('logistic_model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# model.save('my_ft.model')
files.download('logistic_model.pkl')
files.download('scaler.pkl')
# files.download('my_ft.model')

print("Donnnnne")

In [ ]:
xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=3,
    eval_metric='mlogloss',
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    tree_method='hist'
)

print("RandomUnderSampler applied")
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state=42)

# Resample the training data using the FINAL combined array
X_train_balanced, y_train_balanced = rus.fit_resample(X_train_final, y_train)

print("Training XGBoost on balanced data...")

xgb_model.fit(X_train_balanced, y_train_balanced)

y_pred_xgb_raw = xgb_model.predict(X_test_final)

y_pred_final = y_pred_xgb_raw

print(f"\nXGBoost Accuracy: {accuracy_score(y_test, y_pred_final):.2f}")
print("\nXGBoost Classification Report:\n", classification_report(y_test, y_pred_final))

with open('xgboost_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

print("XGBoost model saved. You can predict by adding 1 to the model output.")

files.download('xgboost_model.pkl')
print("Download initiated.")

In [ ]:
import numpy as np

def predict_sentiment(feedback_text):

    cleaned = cleaning(feedback_text)
    no_punc = remove_punc(cleaned)
    tokens = word_tokenize(no_punc)
    no_stop = remove_stopWords(tokens)
    lemmas = lemmatizer_word(no_stop)
    processed_text = " ".join(lemmas)

    text_tfidf = tfidf_vectorizer.transform([processed_text])


    text_len = len(feedback_text)
    words = feedback_text.split()
    word_cnt = len(words)
    lexical = len(set(words)) / word_cnt if word_cnt > 0 else 0
    caps = sum(1 for c in feedback_text if c.isupper()) / (text_len + 1)

    # Note: summary_ratio is set to 0 here as we only have the review text
    extra_features = np.array([[lexical, 0, text_len, caps]])
    extra_scaled = scaler.transform(extra_features)

    final_input = hstack([text_tfidf, extra_scaled])

    lr_pred = lr_model.predict(final_input)[0]
    xgb_pred = xgb_model.predict(final_input)[0]


    class_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

    print(f"Input: {feedback_text}")
    print(f"Logistic Regression Prediction: {class_map[lr_pred]}")
    print(f"XGBoost Prediction: {class_map[xgb_pred]}")


In [ ]:
user_input = r"Didn't fit my version of ipod nano even though the picture seemed to be the correct version One person found this helpful"
predict_sentiment(user_input)

In [ ]:
user_input = r"This is out dated and something i was looking to resell."
predict_sentiment(user_input)

In [ ]:
user_input = r"Terrible. I had them for one day and hate them. Terrible sound quality."
predict_sentiment(user_input)

In [ ]:
user_input = """
These earbuds are pretty weird, to start off. They make a sort of figure 8 shape, with the microphone holding the earbuds together where they'd normally split. They're a bit short, they're just barely long enough to fit in my pants pocket (I usually wear pocket tees, so this wasn't a huge issue with me). The plastic part that most people are talking about is a dust plug that only fits devices iPod Touch and older, which was a problem considering I had a generic brand MP3 player. The best solution is to take some pliers to the plastic part, and to just tear it off (be careful not to damage the jack itself, as long as you avoid hitting it with the pliers it should work).

That said, once these earbuds are modified enough to plug properly into your device, they actually sound very nice. My last earbuds (a $20 pair of Skullcandy earbuds) had about the same sound quality. The bass is a lot nicer than I expected. I haven't done much testing with the mic, but so far I haven't had any problems with it. They also came with two extra pairs of cushions, which was also pretty neat. So far, the only major issue that affected me was that I'd pictured getting a completely different set of earbuds, considering the picture.

To sum everything up:

PROS
-Decent sound quality
-Actual working microphone
-Comes with two extra pairs of cushions

CONS
-Shorter than expected
-Had to remove plastic dust plug to use on MP3
-Not what's shown in the preview picture

**EDIT** About three months after purchasing, they stopped working. Wouldn't recommend for anything other than a short-term fix.
"""
predict_sentiment(user_input)

In [ ]:
df.loc[0]['Text']